# 10. Probar el aplicativo en Colab

Ejecuta la primera celda para conectar la carpeta de Drive. Después escribe tu texto y elige `Solo M2` o `Comparar M0, M1 y M2`. Se utilizan los pesos de la corrida seleccionada en `corrida.json` dentro de la carpeta de Drive.

In [ ]:
# Inicio al abrir este cuaderno directamente en Google Colab.
from pathlib import Path

if not Path("corrida.json").is_file():
    import os
    import subprocess
    import sys

    try:
        from google.colab import drive
    except ImportError as error:
        raise FileNotFoundError("En Jupyter, abre la carpeta que contiene corrida.json.") from error

    proyecto_drive = Path("/content/drive/MyDrive/Tesis_RoBERTuito_M0_M1_M2")
    if not (proyecto_drive / "corrida.json").is_file():
        drive.mount("/content/drive", timeout_ms=180000)
    if not (proyecto_drive / "corrida.json").is_file():
        raise FileNotFoundError("Busca Tesis_RoBERTuito_M0_M1_M2 directamente en Mi unidad.")

    os.chdir(proyecto_drive)
    sys.path.insert(0, str(proyecto_drive))
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(proyecto_drive / "requirements-cuadernos.txt")],
        check=True,
    )
    print("Carpeta de la tesis conectada:", proyecto_drive)


# 1. Preparar la prueba con los pesos locales.
from pathlib import Path
import json
import os
import sys

PROYECTO = Path.cwd()
if not (PROYECTO / "corrida.json").is_file():
    raise FileNotFoundError("Abre Jupyter desde la carpeta que contiene corrida.json.")
corrida = json.loads((PROYECTO / "corrida.json").read_text(encoding="utf-8"))["corrida"]
sys.path.insert(0, str(PROYECTO / "aplicativo"))
os.environ["TESIS_VARIANTE"] = "comparativa"
from nucleo import predecir, validar_texto

def preparar_modelos(modelos):
    for modelo in modelos:
        registro = PROYECTO / "modelos" / corrida / modelo / "ejecucion.json"
        if not registro.is_file():
            raise FileNotFoundError(f"Falta {modelo} en {corrida}. Ejecuta los cuadernos 06–08 para completar esta corrida.")

print("Corrida seleccionada:", corrida)

In [ ]:
# @title 2. Clasificar un texto
texto = "Me alegra que todo saliera bien."  # @param {type:"string"}
modo = "Solo M2"  # @param ["Solo M2", "Comparar M0, M1 y M2"]

import pandas as pd
from IPython.display import display

validar_texto(texto)
modelos = ("M2",) if modo == "Solo M2" else ("M0", "M1", "M2")
preparar_modelos(modelos)
resultados = []
for modelo in modelos:
    resultado = predecir(texto, modelo)
    probabilidades = resultado["porcentajes"]
    resultados.append({
        "Modelo": modelo,
        "Polaridad": resultado["etiqueta"],
        "Negativa (%)": probabilidades["NEG"],
        "Neutral (%)": probabilidades["NEU"],
        "Positiva (%)": probabilidades["POS"],
    })
display(pd.DataFrame(resultados).round(2))

## Opcional. Abrir Gradio

Marca `abrir_gradio = True` en la celda siguiente para abrir el aplicativo dentro de Colab. Se genera un enlace público temporal para la sesión.

In [ ]:
abrir_gradio = False

if abrir_gradio:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROYECTO / "requirements-gradio.txt")], check=True)
    from app import RecursosLocales, construir_app
    from starlette.middleware import Middleware

    preparar_modelos(("M0", "M1", "M2"))
    if "demo" in globals():
        demo.close()
    demo = construir_app()
    demo.launch(share=True, inline=True, ssr_mode=False, show_api=False,
                max_file_size=0, blocked_paths=[str(PROYECTO)],
                app_kwargs={"middleware": [Middleware(RecursosLocales)]})